# Evaluation
Tính toán các chỉ số (BLEU, METEOR, ROUGE, CIDEr, SPICE) và trực quan hóa 5 mẫu thử.

In [ ]:
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import sys

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import compute_metrics
from src.config import PREDICTIONS_PATH, TEST_DF_PATH, IMAGES_PATH

## 1. Tính toán Metrics

In [ ]:
print(f"Bắt đầu chấm điểm dựa trên file: {PREDICTIONS_PATH.name}\n")
metrics = compute_metrics(PREDICTIONS_PATH, TEST_DF_PATH)

print("\n" + "="*30)
print("🏆 FINAL METRICS SCORES 🏆")
print("="*30)
for metric, score in metrics.items():
    print(f"{metric:>10}: {(score*100):.2f}")
print("="*30)

## 2. Visualize Predictions vs Ground Truths

In [ ]:
# Tải lại dự đoán dưới dạng dictionary {imgid: caption}
with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    preds = {int(p["imgid"]): p["caption"] for p in json.load(f)}

# Tải test dataset
test_df = pd.read_parquet(TEST_DF_PATH)

# Lấy ngẫu nhiên 5 ảnh (đổi random_state để xem các ảnh khác)
sample_df = test_df.sample(5, random_state=42)

for _, row in sample_df.iterrows():
    imgid = int(row["imgid"])
    image_path = IMAGES_PATH / row["filepath"] / row["filename"]
    
    predicted_caption = preds.get(imgid, "NO PREDICTION FOUND")
    ground_truths = row["all_raws"]
    
    # Hiển thị ảnh
    plt.figure(figsize=(6, 4))
    try:
        img = Image.open(image_path).convert("RGB")
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    except Exception as e:
        print(f"⚠️ Error loading image {image_path}: {e}")
        continue
    
    # In caption
    print(f"Image ID: {imgid}")
    print(f"Prediction:")
    print(f"   -  {predicted_caption}")
    print(f"Ground Truths:")
    for i, gt in enumerate(ground_truths, 1):
        print(f"   {i}. {gt}")
    print("-" * 80 + "\n")